# 06 Email and web features: do they lift the strong tabular baselines?
Pre-registered in `docs/PROJECT_LOG.md` section 6b (written before any model was fit on these features): feature sets, decision rule, interpretation rule.
Repeat 0 only (5 folds), RF and XGBoost with the fixed notebook-04 hyperparameters, no tuning, shared `make_folds` folds, split by user.
A GO would mean the email/web logs carry signal, not that a graph helps.

In [1]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from itgnn.commslogs import add_comms_features, comms_feature_sets, NEW_BASE
from itgnn.data import load_config, prepare
from itgnn.evaluation import alert_budget, evaluate_scores, make_folds
from itgnn.weekly_features import FEATURE_SETS, add_history_features, drop_warmup
from itgnn.windows import label_weeks, load_attack_windows, make_weekly_features, week_masks

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml")
SEED, QS = cfg["seed"], [0.25, 0.5, 1, 2]
LOG = open(os.environ.get("PROBE_LOG", os.devnull), "a")
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 60
pd.options.display.max_rows = 200

d = prepare(cfg)
users, labels = d["users"], d["labels"]
folds = [f for f in make_folds(users, labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED) if f["repeat"] == 0]
assert len(folds) == 5
windows = load_attack_windows(cfg["raw_dir"])
ins = pd.read_csv(Path(cfg["raw_dir"]) / "answers" / "insiders.csv")
scen = ins[ins["dataset"].astype(str) == "4.2"].set_index("user")["scenario"]
SCEN = sorted(scen.unique())

## 1. Weekly rows, features, and checks against the saved W2 run

In [2]:
full = make_weekly_features(d["logon"], d["device"], d["file"], users)
full["label"] = label_weeks(full, windows)
full["cal_week"] = (full["week_start"] - full["week_start"].min()).dt.days // 7
full = add_history_features(full, d["logon"])           # sorted by (user, week_start), W2 history, before the warm-up drop
email = pd.read_parquet("data/processed/email.parquet")
agg = pd.read_parquet("data/processed/http_user_week_domain.parquet")
full, dropped = add_comms_features(full, email, agg)
weekly = drop_warmup(full)
y = weekly["label"]
print(f"rows {len(full):,} -> {len(weekly):,} after warm-up; attack weeks {int(weekly['label'].sum())}; prevalence {y.mean():.4f}")
print("user-weeks with email / web events but NO Logon row (dropped from the model):", dropped)
print("email events in the dropped weeks are counted as user-weeks, not events.")

saved = pd.read_parquet("results/weekly_v2_oof_scores.parquet", columns=["user", "week_start", "model", "repeat", "fold", "score"])
keys_saved = saved[saved["model"] == "RF W2"].drop_duplicates(["user", "week_start"])[["user", "week_start"]]
k1 = weekly[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True)
k2 = keys_saved.sort_values(["user", "week_start"]).reset_index(drop=True)
pd.testing.assert_frame_equal(k1, k2)
print("row keys identical to the saved W2 run:", len(k1), "user-weeks")
SETS = comms_feature_sets(FEATURE_SETS["W2"])
allcols = sorted({c for v in SETS.values() for c in v})
assert np.isfinite(weekly[allcols].to_numpy(float)).all(), "NaN or inf in features"
print({k: len(v) for k, v in SETS.items()})

rows 67,238 -> 63,238 after warm-up; attack weeks 322; prevalence 0.0051
user-weeks with email / web events but NO Logon row (dropped from the model): {'email': 0, 'web': 0}
email events in the dropped weeks are counted as user-weeks, not events.


row keys identical to the saved W2 run: 63238 user-weeks
{'W2': 16, 'W2+E': 49, 'W2+H': 46, 'W2+E+H': 79, 'W2+E+H generic': 61, 'W2+E+H no-first-seen': 70, 'W2+calendar': 17}


In [3]:
# descriptive sanity checks on the new features (nothing is selected from these)
cov = weekly.groupby(weekly["user"].isin(labels).map({True: "insiders", False: "normal"}))[["em_n", "wb_n"]].agg(lambda s: (s > 0).mean())
print("share of user-weeks with any email / any web activity, by group:"); display(cov)
print("wb_share_rare, mean by calendar-week block (burn-in: domain history is nearly empty at first):")
display(weekly.groupby(pd.cut(weekly["cal_week"], [-1, 3, 7, 12, 25, 50, 80]), observed=True)["wb_share_rare"].agg(["mean", "size"]))
print("category counts: user-weeks with a visit, and distinct users with a visit")
rows = {c: {"user-weeks": int((weekly[c] > 0).sum()), "users": weekly.loc[weekly[c] > 0, "user"].nunique()} for c in [c for c in NEW_BASE if c.startswith("wb_n_") and c.split("wb_n_")[1] in ("leak", "job", "cloud", "download", "competitor")] + ["em_n_competitor_recip"]}
display(pd.DataFrame(rows).T)

share of user-weeks with any email / any web activity, by group:


,em_n,wb_n
user,,
insiders,1.000,1.000
normal,1.000,1.000


wb_share_rare, mean by calendar-week block (burn-in: domain history is nearly empty at first):


,mean,size
cal_week,,
"(3, 7]",0.000,3065
"(7, 12]",0.000,4939
"(12, 25]",0.000,12705
"(25, 50]",0.000,23443
"(50, 80]",0.000,19086


category counts: user-weeks with a visit, and distinct users with a visit


,user-weeks,users
wb_n_leak,52,30
wb_n_job,33942,902
wb_n_cloud,31104,849
wb_n_download,9,9
wb_n_competitor,18942,513
em_n_competitor_recip,51919,999


## 2. Fits (7 sets x RF/XGBoost x 5 folds = 70). Same hyperparameters as notebook 04

In [4]:
def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
def xgb(ytr):
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(),
                         n_jobs=4, random_state=SEED, verbosity=0)
ALGOS = {"RF": lambda ytr: rf(), "XGB": xgb}
t0 = time.time(); metric_rows, oof_parts, fit_times = [], [], []
for f in folds:
    tr, te = week_masks(weekly, f)
    for sname, cols in SETS.items():
        Xtr, Xte = weekly.loc[tr, cols], weekly.loc[te, cols]
        for a, mk in ALGOS.items():
            ts = time.time()
            s = mk(y[tr]).fit(Xtr, y[tr]).predict_proba(Xte)[:, 1]
            assert np.isfinite(s).all()
            ev = evaluate_scores(y[te].to_numpy(), s, k=100)
            name = f"{a} {sname}"
            metric_rows.append({"model": name, "algo": a, "set": sname, "repeat": f["repeat"], "fold": f["fold"], **ev})
            fit_times.append(time.time() - ts)
            oof_parts.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(),
                                           "cal_week": weekly.loc[te, "cal_week"].to_numpy(), "repeat": f["repeat"], "fold": f["fold"],
                                           "model": name, "score": s, "label": y[te].to_numpy()}))
    print(f"fold {f['fold']} done, elapsed {time.time() - t0:.0f}s", file=LOG, flush=True)
results = pd.DataFrame(metric_rows)
oof = pd.concat(oof_parts, ignore_index=True)
oof.to_parquet("results/comms_probe_oof_scores.parquet", index=False)
print(f"70 fits in {time.time() - t0:.0f}s (mean fit {np.mean(fit_times):.1f}s, max {np.max(fit_times):.1f}s); saved results/comms_probe_oof_scores.parquet {oof.shape}")

70 fits in 103s (mean fit 1.5s, max 7.0s); saved results/comms_probe_oof_scores.parquet (885332, 8)


## 3. Reproduction check: W2 on repeat 0 must match the saved step-4 scores

In [5]:
rep = []
for a in ["RF", "XGB"]:
    sv = saved[(saved["model"] == f"{a} W2") & (saved["repeat"] == 0)]
    for fo, g in oof[oof["model"] == f"{a} W2"].groupby("fold"):
        m = g.merge(sv[sv["fold"] == fo], on=["user", "week_start"], suffixes=("", "_saved"))
        assert len(m) == len(g)
        new = evaluate_scores(m["label"], m["score"], 100)["pr_auc"]; old = evaluate_scores(m["label"], m["score_saved"], 100)["pr_auc"]
        rep.append({"algo": a, "fold": fo, "pr_auc now": new, "pr_auc saved": old, "abs diff": abs(new - old), "max score diff": (m["score"] - m["score_saved"]).abs().max()})
rep = pd.DataFrame(rep); display(rep)
print("max PR-AUC diff RF:", rep.loc[rep.algo == "RF", "abs diff"].max(), "| XGB:", rep.loc[rep.algo == "XGB", "abs diff"].max())

,algo,fold,pr_auc now,pr_auc saved,abs diff,max score diff
0,RF,0,0.463,0.463,0.000,0.000
1,RF,1,0.626,0.626,0.000,0.000
2,RF,2,0.468,0.468,0.000,0.000
3,RF,3,0.477,0.477,0.000,0.000
4,RF,4,0.460,0.460,0.000,0.000
5,XGB,0,0.435,0.435,0.000,0.000
6,XGB,1,0.617,0.617,0.000,0.000
7,XGB,2,0.414,0.414,0.000,0.000
8,XGB,3,0.497,0.497,0.000,0.000
9,XGB,4,0.362,0.362,0.000,0.000


max PR-AUC diff RF: 0.0 | XGB: 0.0


## 4. PR-AUC and paired differences vs W2 (same folds)

In [6]:
S = results.groupby("model")[["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]].agg(["mean", "std"])
order = [f"{a} {s}" for a in ["RF", "XGB"] for s in SETS]
display(pd.DataFrame({m: S[m].apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1) for m in ["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]}).loc[order])
pv = results.pivot_table(index="fold", columns="model", values="pr_auc")
print("per-fold PR-AUC:"); display(pv[order])
rows = {}
for a in ["RF", "XGB"]:
    for s in list(SETS)[1:]:
        dl = pv[f"{a} {s}"] - pv[f"{a} W2"]
        rows[f"{a}: {s} - W2"] = {"mean diff": dl.mean(), "std": dl.std(), "folds positive": f"{int((dl > 0).sum())}/5", **{f"fold {i}": dl[i] for i in dl.index}}
paired = pd.DataFrame(rows).T; display(paired)

,pr_auc,roc_auc,precision_at_k,recall_at_k
model,,,,
RF W2,0.499 ± 0.072,0.979 ± 0.018,0.352 ± 0.022,0.551 ± 0.073
RF W2+E,0.575 ± 0.122,0.975 ± 0.023,0.408 ± 0.081,0.640 ± 0.152
RF W2+H,0.772 ± 0.101,0.998 ± 0.002,0.544 ± 0.053,0.851 ± 0.126
RF W2+E+H,0.743 ± 0.098,0.997 ± 0.002,0.516 ± 0.078,0.807 ± 0.148
RF W2+E+H generic,0.675 ± 0.127,0.987 ± 0.011,0.450 ± 0.086,0.707 ± 0.171
RF W2+E+H no-first-seen,0.709 ± 0.094,0.996 ± 0.003,0.494 ± 0.080,0.773 ± 0.155
RF W2+calendar,0.590 ± 0.105,0.986 ± 0.015,0.420 ± 0.058,0.661 ± 0.148
XGB W2,0.465 ± 0.097,0.986 ± 0.007,0.346 ± 0.030,0.542 ± 0.082
XGB W2+E,0.566 ± 0.107,0.987 ± 0.008,0.406 ± 0.049,0.636 ± 0.107


per-fold PR-AUC:


model,RF W2,RF W2+E,RF W2+H,RF W2+E+H,RF W2+E+H generic,RF W2+E+H no-first-seen,RF W2+calendar,XGB W2,XGB W2+E,XGB W2+H,XGB W2+E+H,XGB W2+E+H generic,XGB W2+E+H no-first-seen,XGB W2+calendar
fold,,,,,,,,,,,,,,
0,0.463,0.594,0.744,0.741,0.700,0.679,0.521,0.435,0.587,0.834,0.841,0.788,0.846,0.515
1,0.626,0.728,0.776,0.833,0.779,0.834,0.765,0.617,0.719,0.904,0.928,0.916,0.947,0.741
2,0.468,0.644,0.935,0.773,0.774,0.723,0.600,0.414,0.596,0.984,0.979,0.939,0.979,0.497
3,0.477,0.420,0.659,0.579,0.468,0.574,0.563,0.497,0.474,0.783,0.769,0.658,0.764,0.615
4,0.460,0.489,0.746,0.787,0.654,0.736,0.500,0.362,0.453,0.870,0.878,0.819,0.850,0.456


,mean diff,std,folds positive,fold 0,fold 1,fold 2,fold 3,fold 4
RF: W2+E - W2,0.076,0.091,4/5,0.130,0.101,0.176,-0.057,0.029
RF: W2+H - W2,0.273,0.124,5/5,0.280,0.150,0.467,0.182,0.286
RF: W2+E+H - W2,0.244,0.091,5/5,0.278,0.207,0.305,0.102,0.327
RF: W2+E+H generic - W2,0.176,0.118,4/5,0.237,0.152,0.306,-0.009,0.194
RF: W2+E+H no-first-seen - W2,0.210,0.069,5/5,0.216,0.207,0.255,0.097,0.276
RF: W2+calendar - W2,0.091,0.044,5/5,0.058,0.139,0.132,0.086,0.040
XGB: W2+E - W2,0.101,0.078,4/5,0.152,0.102,0.181,-0.022,0.090
XGB: W2+H - W2,0.410,0.128,5/5,0.399,0.288,0.569,0.287,0.508
XGB: W2+E+H - W2,0.414,0.126,5/5,0.407,0.311,0.564,0.272,0.515
XGB: W2+E+H generic - W2,0.359,0.141,5/5,0.353,0.299,0.524,0.162,0.457


## 5. Alert budget and per scenario (repeat 0, one repeat so no spread)

In [7]:
scen_of = scen.to_dict()
bud = alert_budget(oof, labels, scen_of, QS, SEED)
elig = weekly[weekly["label"] == 1].groupby("user").size().index
print("eligible insiders per scenario:", {int(s): int((scen[list(elig)] == s).sum()) for s in SCEN}, "| normal users:", int(weekly.loc[~weekly["user"].isin(labels), "user"].nunique()))
cols = ["caught", "normal flagged"] + [f"caught s{s}" for s in SCEN]
b = bud.set_index(["model", "q%"])[cols]
for a in ["RF", "XGB"]:
    print(f"--- {a}: insiders caught (of 70 eligible), normal users flagged (of 930), caught per scenario"); display(b.loc[[f"{a} {s}" for s in SETS]])

eligible insiders per scenario: {1: 30, 2: 30, 3: 10} | normal users: 930
--- RF: insiders caught (of 70 eligible), normal users flagged (of 930), caught per scenario


caught  normal flagged  caught s1  caught s2  caught s3
model                   q%                                                            
RF W2                   0.250      49               3         22         23          4
                        0.500      54              10         26         24          4
                        1.000      60              19         28         26          6
                        2.000      65              41         29         29          7
RF W2+E                 0.250      43               2         19         20          4
                        0.500      54               8         26         22          6
                        1.000      57              17         28         23          6
                        2.000      60              46         28         25          7
RF W2+H                 0.250      39               1         17         20          2
                        0.500      60               8         26         28          6
                        1.000      67              26         30         29          8
                        2.000      70              79         30         30         10
RF W2+E+H               0.250      41               2         19         19          3
                        0.500      58               6         28         24          6
                        1.000      66              20         30         29          7
                        2.000      70              67         30         30         10
RF W2+E+H generic       0.250      44               3         18         20          6
                        0.500      54               7         24         23          7
                        1.000      60              23         28         24          8
                        2.000      65              54         29         28          8
RF W2+E+H no-first-seen 0.250      38               2         17         18          3
                        0.500      59               6         28         26          5
                        1.000      65              23         30         28          7
                        2.000      67              56         30         29          8
RF W2+calendar          0.250      45               2         19         23          3
                        0.500      52               8         25         24          3
                        1.000      60              29         28         27          5
                        2.000      67              73         30         30          7

--- XGB: insiders caught (of 70 eligible), normal users flagged (of 930), caught per scenario


caught  normal flagged  caught s1  caught s2  caught s3
model                    q%                                                            
XGB W2                   0.250      51               2         22         25          4
                         0.500      53               5         24         25          4
                         1.000      58              12         27         27          4
                         2.000      62              36         29         29          4
XGB W2+E                 0.250      41               1         16         22          3
                         0.500      52               8         24         24          4
                         1.000      57              23         27         26          4
                         2.000      62              44         29         29          4
XGB W2+H                 0.250      56               1         30         24          2
                         0.500      62               6         30         29          3
                         1.000      67              26         30         29          8
                         2.000      69              62         30         30          9
XGB W2+E+H               0.250      56               0         30         24          2
                         0.500      62               7         30         28          4
                         1.000      68              27         30         30          8
                         2.000      69              51         30         30          9
XGB W2+E+H generic       0.250      50               1         20         26          4
                         0.500      59              11         27         28          4
                         1.000      66              23         30         29          7
                         2.000      67              63         30         29          8
XGB W2+E+H no-first-seen 0.250      54               0         28         22          4
                         0.500      63               9         30         29          4
                         1.000      66              24         30         29          7
                         2.000      68              49         30         30          8
XGB W2+calendar          0.250      44               2         18         24          2
                         0.500      50               7         22         25          3
                         1.000      56              26         25         28          3
                         2.000      62              64         29         29          4

## 6. Burn-in check (descriptive): PR-AUC restricted to calendar week >= 13, from the saved predictions

In [8]:
bi = []
for (m, fo), g in oof.groupby(["model", "fold"]):
    g = g[g["cal_week"] >= 13]
    bi.append({"model": m, "fold": fo, "pr_auc_w13": evaluate_scores(g["label"], g["score"], 100)["pr_auc"]})
bi = pd.DataFrame(bi); pb = bi.pivot_table(index="fold", columns="model", values="pr_auc_w13")
print("rows kept: weeks >= 13;  attack weeks:", int(oof[(oof['model'] == 'RF W2') & (oof['cal_week'] >= 13)]['label'].sum()), "of", int(oof[oof['model'] == 'RF W2']['label'].sum()))
t = pd.DataFrame({"PR-AUC wk>=13 (mean ± std)": pb[order].apply(lambda s: f"{s.mean():.3f} ± {s.std():.3f}")})
for a in ["RF", "XGB"]:
    for s in list(SETS)[1:]:
        t.loc[f"{a} {s}", "paired diff vs W2"] = (pb[f"{a} {s}"] - pb[f"{a} W2"]).mean()
display(t.loc[order])

rows kept: weeks >= 13;  attack weeks: 322 of 322


,PR-AUC wk>=13 (mean ± std),paired diff vs W2
model,,
RF W2,0.527 ± 0.079,NaN
RF W2+E,0.587 ± 0.127,0.059
RF W2+H,0.772 ± 0.101,0.245
RF W2+E+H,0.743 ± 0.097,0.216
RF W2+E+H generic,0.679 ± 0.127,0.152
RF W2+E+H no-first-seen,0.711 ± 0.094,0.183
RF W2+calendar,0.590 ± 0.105,0.063
XGB W2,0.500 ± 0.099,NaN
XGB W2+E,0.579 ± 0.115,0.079


## 7. Pre-registered decision rule (docs/PROJECT_LOG.md 6b)

In [9]:
def dl(a, s): return pv[f"{a} {s}"] - pv[f"{a} W2"]
rf_d, xg_d = dl("RF", "W2+E+H"), dl("XGB", "W2+E+H")
q5 = bud[bud["q%"] == 0.5].set_index("model")
dc = q5.loc["RF W2+E+H", "caught"] - q5.loc["RF W2", "caught"]; dn = q5.loc["RF W2+E+H", "normal flagged"] - q5.loc["RF W2", "normal flagged"]
cal = dl("RF", "W2+calendar").mean()
crit = {
 "1. RF mean delta PR-AUC >= +0.05": (rf_d.mean(), rf_d.mean() >= 0.05),
 "2. RF >= 4 of 5 folds positive": (int((rf_d > 0).sum()), (rf_d > 0).sum() >= 4),
 "3. XGB mean delta >= +0.03 and >= 4/5 folds positive": (f"{xg_d.mean():.3f}, {int((xg_d > 0).sum())}/5", xg_d.mean() >= 0.03 and (xg_d > 0).sum() >= 4),
 "4. q=0.5%: RF +>=3 insiders caught, <=2 extra normal flagged": (f"+{dc:g} caught, {dn:+g} normal", dc >= 3 and dn <= 2),
 "5. calendar control delta < +0.01": (cal, cal < 0.01),
}
display(pd.DataFrame(crit, index=["value", "met"]).T)
if all(v[1] for v in crit.values()): verdict = "GO"
elif rf_d.mean() < 0.02: verdict = "NO-GO"
else: verdict = "AMBIGUOUS"
print("VERDICT:", verdict, "| RF mean delta", round(rf_d.mean(), 4), "| XGB mean delta", round(xg_d.mean(), 4))
gen = dl("RF", "W2+E+H generic").mean()
print("generic RF mean delta vs W2:", round(gen, 4),
      "->", "named-site only (generic < +0.02): do NOT build a graph" if gen < 0.02 else "case for a relational graph (generic >= +0.03)" if gen >= 0.03 else "grey zone (0.02-0.03), no conclusion")
top = results[results["model"] == "RF W2+E+H"]["pr_auc"].mean()
print("W2+E+H RF PR-AUC", round(top, 3), "-> leakage audit REQUIRED" if top > 0.80 else "-> below the 0.80 leakage-audit trigger")
print("NOTE: wikileaks.org has exactly 30 users and scenario 1 has 30 insiders; a scenario-1 gain is expected to be easy here.")

,value,met
1. RF mean delta PR-AUC >= +0.05,0.244,True
2. RF >= 4 of 5 folds positive,5,True
3. XGB mean delta >= +0.03 and >= 4/5 folds positive,"0.414, 5/5",True
"4. q=0.5%: RF +>=3 insiders caught, <=2 extra normal flagged","+4 caught, -4 normal",True
5. calendar control delta < +0.01,0.091,False


VERDICT: AMBIGUOUS | RF mean delta 0.2436 | XGB mean delta 0.4139
generic RF mean delta vs W2: 0.1757 -> case for a relational graph (generic >= +0.03)
W2+E+H RF PR-AUC 0.743 -> below the 0.80 leakage-audit trigger
NOTE: wikileaks.org has exactly 30 users and scenario 1 has 30 insiders; a scenario-1 gain is expected to be easy here.


## 8. Descriptive: RF impurity importances for W2+E+H (fit on fold-0 training users only)

In [10]:
tr0, _ = week_masks(weekly, folds[0])
m = rf().fit(weekly.loc[tr0, SETS["W2+E+H"]], y[tr0])
imp = pd.Series(m.feature_importances_, index=SETS["W2+E+H"]).sort_values(ascending=False)
display(imp.head(20).to_frame("importance"))
print("share of total importance on named-site columns:", round(imp[[c for c in imp.index if any(k in c for k in ["_leak", "_job", "_cloud", "_download", "_competitor"])]].sum(), 3))

,importance
n_device,0.147
hist_mean_n_file,0.118
hist_mean_n_device,0.107
n_file,0.107
dev_n_device,0.063
hist_mean_wb_n_competitor,0.046
dev_n_file,0.042
dev_wb_n_competitor,0.028
hist_mean_wb_n_job,0.024
dev_wb_n_leak,0.020


share of total importance on named-site columns: 0.198
